<a href="https://colab.research.google.com/github/tzekensoh/camp-python/blob/main/hanoi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# %% [markdown]
# # Tower of Hanoi – Visual Demo
#
# This notebook shows the Tower of Hanoi puzzle solved step by step with a graphical animation.

# %%
# Run this first
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib import rc
from IPython.display import HTML, display

# For Colab: render animations inline with interactive widgets
rc('animation', html='jshtml')

In [ ]:
# %% [markdown]
## Step 1: Puzzle setup

# %%
PEG_NAMES = ['A', 'B', 'C']
NUM_DISKS = 4          # You can reduce to 3 for a shorter demo
PEG_X = {0: -2, 1: 0, 2: 2}
DISK_COLORS = ['#4C78A8', '#F58518', '#54A24B', '#E45756', '#72B7B2', '#B279A2', '#FF9DA6']

In [ ]:
# %% [markdown]
## Step 2: Recursive Tower of Hanoi

# %%
def hanoi(n, source, target, auxiliary, moves=None):
    if moves is None:
        moves = []
    if n == 1:
        moves.append((source, target))
    else:
        hanoi(n-1, source, auxiliary, target, moves)
        moves.append((source, target))
        hanoi(n-1, auxiliary, target, source, moves)
    return moves

moves = hanoi(NUM_DISKS, 'A', 'C', 'B')
print(f"Number of moves: {len(moves)}")
print(f"First few moves: {moves[:5]}")

In [ ]:
# %% [markdown]
## Step 3: Tower state representation

# %%
state = {
    'A': list(range(NUM_DISKS, 0, -1)),  # peg A has all disks
    'B': [],                              # peg B starts empty
    'C': []                               # peg C starts empty
}
print(state)

In [ ]:
# %% [markdown]
## Step 4: Drawing one frame

# %%
def draw_state(ax, state, title='Tower of Hanoi'):
    ax.clear()
    ax.set_xlim(-3, 3)
    ax.set_ylim(-0.5, NUM_DISKS + 1.5)
    ax.axis('off')
    ax.set_title(title)

    for x in PEG_X.values():
        ax.plot([x, x], [0, NUM_DISKS + 0.2], color='black', lw=4)

    for peg_name, x in PEG_X.items():
        ax.text(x, -0.25, peg_name, ha='center', va='top', fontsize=12)

    for peg_name, disks in state.items():
        x = PEG_X['ABC'.index(peg_name)]
        for level, disk in enumerate(disks):
            width = 0.4 + 0.25 * disk
            y = level + 0.1
            color = DISK_COLORS[(disk - 1) % len(DISK_COLORS)]
            rect = Rectangle(
                (x - width/2, y),
                width, 0.6,
                facecolor=color,
                edgecolor='black'
            )
            ax.add_patch(rect)
            ax.text(
                x, y + 0.3,
                str(disk),
                ha='center', va='center',
                color='white',
                fontsize=10,
                weight='bold'
            )

In [ ]:
# %% [markdown]
## Step 5: Static picture at the start

# %%
fig, ax = plt.subplots(figsize=(7, 4))
draw_state(ax, state, 'Starting position')
plt.show()

In [ ]:
# %% [markdown]
## Step 6: Generate animation frames

# %%
frames = []
current = {k: v.copy() for k, v in state.items()}
frames.append({k: v.copy() for k, v in current.items()})  # initial frame

for src, dst in moves:
    disk = current[src].pop()
    current[dst].append(disk)
    frames.append({k: v.copy() for k, v in current.items()})

print(f"Total frames: {len(frames)}")

In [ ]:
# %% [markdown]
## Step 7: Play the animation

# %%
fig, ax = plt.subplots(figsize=(7, 4))

def update(frame_index):
    draw_state(ax, frames[frame_index], f'Step {frame_index} of {len(frames)-1}')

anim = FuncAnimation(fig, update, frames=len(frames), interval=900, repeat=False)
HTML(anim.to_jshtml())